In [1]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST
import matplotlib.pyplot as plt
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

nll_loss 的全称是 Negative Log Likelihood Loss（负对数似然损失）
数学公式简写：$$loss(x, class) = -x[class]$$其中 $x$ 是模型的输出，$class$ 是真实的标签索引。

In [2]:
class Net(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = torch.nn.Linear(28 * 28, 128)
        self.fc2 = torch.nn.Linear(128, 128)
        self.fc3 = torch.nn.Linear(128, 128)
        self.fc4 = torch.nn.Linear(128, 10)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x) # 输出原始得分 (Raw Logits)
        return x 
    
class Net2(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        self.conv2 = torch.nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.conv3 = torch.nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.fc1 = torch.nn.Linear(64 * 3 * 3, 128)
        self.fc2 = torch.nn.Linear(128, 10)

    def forward(self, x):
        x = F.max_pool2d(torch.relu(self.conv1(x)), kernel_size=2, stride=2)
        x = F.max_pool2d(torch.relu(self.conv2(x)), kernel_size=2, stride=2)
        x = F.max_pool2d(torch.relu(self.conv3(x)), kernel_size=2, stride=2)
        x = x.view(-1, 64 * 3 * 3)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def get_data_loader(is_train):
    to_tensor = transforms.Compose([transforms.ToTensor()])
    dataset = MNIST("./data", train=is_train, transform=to_tensor, download=True)
    return DataLoader(dataset, batch_size=16, shuffle=True) 

def evaluate(test_data, net):
    n_correct = 0
    n_total = 0
    with torch.no_grad():
        for (x, y) in test_data:
            x, y = x.to(device), y.to(device)
            outputs = net(x.view(-1, 28 * 28))
            # 优化：使用向量化计算代替 for 循环，大幅提升速度
            predictions = torch.argmax(outputs, dim=1)
            n_correct += (predictions == y).sum().item()
            n_total += y.size(0)

    return n_correct / n_total


def evaluate2(test_data, net):
    n_correct = 0
    n_total = 0
    with torch.no_grad():
        for (x, y) in test_data:
            x, y = x.to(device), y.to(device)
            outputs = net(x)
            predictions = torch.argmax(outputs, dim=1)
            n_correct += (predictions == y).sum().item()
            n_total += y.size(0)

    return n_correct / n_total

# def main():
#     train_data = get_data_loader(is_train=True)
#     test_data = get_data_loader(is_train=False)
#     net = Net().to(device)
    
#     print("initial accuracy:", evaluate(test_data, net))
#     optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
    
#     # 优化：对于简单的 MNIST 任务，10 个 epoch 已经足够达到很高的准确率了
#     for epoch in range(20): 
#         for (x, y) in train_data:
#             x, y = x.to(device), y.to(device)
#             net.zero_grad()
#             output = net.forward(x.view(-1, 28 * 28))
            
#             # 核心修复：使用 cross_entropy 替代 nll_loss
#             loss = F.cross_entropy(output, y) 
            
#             loss.backward()
#             optimizer.step()
            
#         print("epoch", epoch, "accuracy:", evaluate(test_data, net))

#     # 可视化部分
#     for (n, (x, _)) in enumerate(test_data):
#         if n > 3:
#             break
#         x_gpu = x[0].to(device).view(-1, 28 * 28)

#         predict = torch.argmax(net.forward(x_gpu))
#         plt.figure(n)
#         plt.imshow(x[0].view(28, 28), cmap='gray') # 加上 cmap='gray' 显示灰度图效果更好
#         plt.title("prediction: " + str(int(predict)))
        
#     plt.show()


def main():
    train_data = get_data_loader(is_train=True)
    test_data = get_data_loader(is_train=False)
    net = Net2().to(device)
    
    print("initial accuracy:", evaluate2(test_data, net))
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
    
    for epoch in range(10): 
        for (x, y) in train_data:
            x, y = x.to(device), y.to(device)
            net.zero_grad()
            output = net.forward(x)
            loss = F.cross_entropy(output, y) 
            loss.backward()
            optimizer.step()
            
        print("epoch", epoch, "accuracy:", evaluate2(test_data, net))

if __name__ == "__main__":
    main()

initial accuracy: 0.106
epoch 0 accuracy: 0.9825
epoch 1 accuracy: 0.9851
epoch 2 accuracy: 0.991
epoch 3 accuracy: 0.9883
epoch 4 accuracy: 0.985
epoch 5 accuracy: 0.9889
epoch 6 accuracy: 0.9896
epoch 7 accuracy: 0.9919
epoch 8 accuracy: 0.9932
epoch 9 accuracy: 0.9911
